# T7-31 to T7-33: Clean, Encode, Scale, Split and Hand Over
**Project:** Customer Churn Analysis, Project Team 7 (WILDA26_09)  
**Owner:** Harjot Kaur, Data Engineer  
**Jira tasks:** T7-31 Clean and encode data; T7-32 Scale and split data; T7-33 Milestone: clean data handed over  
**Input:** Dataset_ATS_v2.csv (raw file, checked in T7-30)

This notebook continues from the T7-30 quality check. It applies the cleaning decisions, encodes the categorical fields, scales the numeric fields, creates the stratified train, validation and test split, validates the outputs and saves the files handed over to Dilina (clustering) and Shivani (ANN). Every change is written to a cleaning log.

**Repository location:** `Data_Preparation/preprocessed_data/`

**How to run:** set `RAW_PATH` in the setup cell to the location of `Dataset_ATS_v2.csv` (by default `../raw_data/Dataset_ATS_v2.csv`; in Colab, upload it to the session files and change the path) and select Run All. The cleaned, encoded and clustering files and the cleaning log are written to this folder (`preprocessed_data`); the train, validation and test sets and the scaler parameters are written to `../train_test_sets`.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RAW_PATH = '../raw_data/Dataset_ATS_v2.csv'   # Data_Preparation/raw_data
PREP_DIR = '.'                      # Data_Preparation/preprocessed_data
SPLIT_DIR = '../train_test_sets'    # Data_Preparation/train_test_sets
os.makedirs(SPLIT_DIR, exist_ok=True)
log = []
def add_log(step, action, records, reason):
    log.append({'step': len(log) + 1, 'task': step, 'action': action, 'records_affected': records, 'reason': reason})

## 2. Load the raw data (raw copy kept untouched)

In [2]:
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()
print('Rows:', df.shape[0], '| Columns:', df.shape[1])
add_log('T7-31', 'Loaded raw file; working copy created, raw copy kept unchanged', len(df), 'Keeps the original available for checking')

Rows: 7043 | Columns: 10


## 3. T7-31 Cleaning decisions

**Decision 1: inconsistent MultipleLines records are recoded to No.** A customer without a phone service cannot hold multiple phone lines, so these values are treated as recording errors. The rows are kept because all their other values are valid.

In [3]:
mask = (df['PhoneService'] == 'No') & (df['MultipleLines'] == 'Yes')
print('Records recoded:', mask.sum())
df.loc[mask, 'MultipleLines'] = 'No'
add_log('T7-31', 'MultipleLines recoded from Yes to No where PhoneService = No', int(mask.sum()), 'Multiple lines are not possible without a phone service')
pd.crosstab(df['PhoneService'], df['MultipleLines'])

Records recoded: 260


MultipleLines,No,Yes
PhoneService,,
No,682,0
Yes,3650,2711


**Decision 2: the 302 exact duplicate rows are kept.** The checks below show why. The file has no customer ID and only ten coarse fields, so identical rows are expected among customers with the most common profile. The duplicates are concentrated among new customers on the lowest monthly charges, and their churn rate matches the dataset as a whole, which points to different customers who share the same values rather than repeated records. Removing them would under-represent the most common customer type.

In [4]:
dup_all = df_raw[df_raw.duplicated(keep=False)]
summary = pd.DataFrame({
    'All customers': [len(df_raw), round((df_raw['tenure'] <= 3).mean() * 100, 1), df_raw['MonthlyCharges'].median(), round((df_raw['Churn'] == 'Yes').mean() * 100, 1)],
    'Rows in duplicate groups': [len(dup_all), round((dup_all['tenure'] <= 3).mean() * 100, 1), dup_all['MonthlyCharges'].median(), round((dup_all['Churn'] == 'Yes').mean() * 100, 1)]},
    index=['Rows', 'Tenure 3 months or less (%)', 'Median monthly charges ($)', 'Churn rate (%)'])
print('Exact duplicate rows (beyond first copy):', df_raw.duplicated().sum())
add_log('T7-31', 'Exact duplicate rows kept', int(df_raw.duplicated().sum()), 'No customer ID; duplicates match the most common profile and the overall churn rate')
summary

Exact duplicate rows (beyond first copy): 302


,All customers,Rows in duplicate groups
Rows,7043.0,541.0
Tenure 3 months or less (%),15.1,44.7
Median monthly charges ($),70.0,20.0
Churn rate (%),26.5,25.7


**Decision 3: the 11 customers with zero tenure are kept** as valid new customers. Their other fields are complete and within normal ranges.

In [5]:
zero = df[df['tenure'] == 0]
print('Zero-tenure customers:', len(zero), '| Churned:', (zero['Churn'] == 'Yes').sum())
add_log('T7-31', 'Zero-tenure customers kept as new customers', len(zero), 'All other values valid')
df.to_csv(f'{PREP_DIR}/ats_cleaned.csv', index=False)
print('Saved preprocessed_data/ats_cleaned.csv', df.shape)

Zero-tenure customers: 11 | Churned: 0
Saved preprocessed_data/ats_cleaned.csv (7043, 10)


## 4. T7-31 Encoding

In [6]:
enc = df.copy()
binary_map = {'gender': 'Male', 'Dependents': 'Yes', 'PhoneService': 'Yes', 'MultipleLines': 'Yes', 'Churn': 'Yes'}
for col, positive in binary_map.items():
    enc[col] = (enc[col] == positive).astype(int)
enc = pd.get_dummies(enc, columns=['InternetService', 'Contract'], dtype=int)
enc.columns = [c.replace(' ', '_').replace('-', '_') for c in enc.columns]
cols = [c for c in enc.columns if c != 'Churn'] + ['Churn']
enc = enc[cols]
add_log('T7-31', 'Binary fields coded 1/0 (gender: Male = 1; others: Yes = 1)', len(enc), 'Models need numeric inputs')
add_log('T7-31', 'InternetService (2) and Contract (3) one-hot encoded', len(enc), 'Categories have no natural order')
enc.to_csv(f'{PREP_DIR}/ats_encoded.csv', index=False)
print('Encoded columns:', enc.shape[1]); print(list(enc.columns))
enc.head()

Encoded columns: 13
['gender', 'SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'MonthlyCharges', 'InternetService_DSL', 'InternetService_Fiber_optic', 'Contract_Month_to_month', 'Contract_One_year', 'Contract_Two_year', 'Churn']


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,MonthlyCharges,InternetService_DSL,InternetService_Fiber_optic,Contract_Month_to_month,Contract_One_year,Contract_Two_year,Churn
0,0,0,0,1,0,0,25,1,0,1,0,0,1
1,1,0,0,41,1,0,25,1,0,0,1,0,0
2,0,0,1,52,1,0,19,1,0,1,0,0,0
3,0,0,0,1,1,0,76,1,0,0,1,0,1
4,1,0,0,67,1,0,51,0,1,1,0,0,0


## 5. T7-32 Stratified split (70 / 15 / 15)

In [7]:
X = enc.drop(columns='Churn'); y = enc['Churn']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)
split_check = pd.DataFrame({'rows': [len(y_train), len(y_val), len(y_test)],
                            'share_pct': [round(len(s) / len(y) * 100, 1) for s in (y_train, y_val, y_test)],
                            'churn_rate_pct': [round(s.mean() * 100, 1) for s in (y_train, y_val, y_test)]},
                           index=['train', 'validation', 'test'])
add_log('T7-32', 'Stratified split 70/15/15 on Churn (random_state 42)', len(enc), 'Keeps the 26.5% churn rate in every set')
split_check

,rows,share_pct,churn_rate_pct
train,4930,70.0,26.5
validation,1056,15.0,26.5
test,1057,15.0,26.6


## 6. T7-32 Scaling

The scaler is fitted on the training set only and then applied to the validation and test sets, so no information from those sets leaks into training. Only `tenure` and `MonthlyCharges` are scaled; the 0/1 columns stay as they are.

In [8]:
num_cols = ['tenure', 'MonthlyCharges']
scaler = StandardScaler().fit(X_train[num_cols])
def scale(frame):
    out = frame.copy(); out[num_cols] = scaler.transform(frame[num_cols]); return out
train = scale(X_train).assign(Churn=y_train); val = scale(X_val).assign(Churn=y_val); test = scale(X_test).assign(Churn=y_test)
params = pd.DataFrame({'mean': scaler.mean_.round(4), 'std': np.sqrt(scaler.var_).round(4)}, index=num_cols)
add_log('T7-32', 'StandardScaler fitted on training set; applied to train, validation and test', len(enc), 'Prevents data leakage')
for name, part in [('train', train), ('val', val), ('test', test)]:
    part.to_csv(f'{SPLIT_DIR}/ats_{name}.csv', index=False)
params.to_csv(f'{SPLIT_DIR}/scaler_params.csv')
params

,mean,std
tenure,32.3105,24.5833
MonthlyCharges,64.5901,30.1506


## 7. T7-32 Clustering-ready file

Clustering uses the whole dataset rather than a split, so a separate file is prepared with `tenure` and `MonthlyCharges` standardised on all 7,043 rows. `Churn` is kept as the last column for profiling only and must not be used as a clustering feature.

In [9]:
clus = enc.copy()
clus[num_cols] = StandardScaler().fit_transform(enc[num_cols])
clus.to_csv(f'{PREP_DIR}/ats_clustering_ready.csv', index=False)
add_log('T7-32', 'Clustering-ready file created (numeric fields standardised on full data)', len(clus), 'Clustering is unsupervised and uses all records')
print('Saved preprocessed_data/ats_clustering_ready.csv', clus.shape)

Saved preprocessed_data/ats_clustering_ready.csv (7043, 13)


## 8. T7-33 Final validation before handover

In [10]:
checks = []
for name, part in [('encoded', enc), ('train', train), ('val', val), ('test', test), ('clustering', clus)]:
    checks.append({'file': name, 'rows': len(part), 'columns': part.shape[1], 'missing': int(part.isna().sum().sum()),
                   'non_numeric_cols': int((part.dtypes == 'object').sum()), 'churn_pct': round(part['Churn'].mean() * 100, 1)})
checks = pd.DataFrame(checks)
assert checks['missing'].sum() == 0 and checks['non_numeric_cols'].sum() == 0
assert len(train) + len(val) + len(test) == len(enc)
assert ((df['PhoneService'] == 'No') & (df['MultipleLines'] == 'Yes')).sum() == 0
print('All validation checks passed')
checks

All validation checks passed


,file,rows,columns,missing,non_numeric_cols,churn_pct
0,encoded,7043,13,0,0,26.5
1,train,4930,13,0,0,26.5
2,val,1056,13,0,0,26.5
3,test,1057,13,0,0,26.6
4,clustering,7043,13,0,0,26.5


In [11]:
log_df = pd.DataFrame(log)
log_df.to_csv(f'{PREP_DIR}/cleaning_log.csv', index=False)
log_df

,step,task,action,records_affected,reason
0,1,T7-31,"Loaded raw file; working copy created, raw cop...",7043,Keeps the original available for checking
1,2,T7-31,MultipleLines recoded from Yes to No where Pho...,260,Multiple lines are not possible without a phon...
2,3,T7-31,Exact duplicate rows kept,302,No customer ID; duplicates match the most comm...
3,4,T7-31,Zero-tenure customers kept as new customers,11,All other values valid
4,5,T7-31,Binary fields coded 1/0 (gender: Male = 1; oth...,7043,Models need numeric inputs
5,6,T7-31,InternetService (2) and Contract (3) one-hot e...,7043,Categories have no natural order
6,7,T7-32,Stratified split 70/15/15 on Churn (random_sta...,7043,Keeps the 26.5% churn rate in every set
7,8,T7-32,StandardScaler fitted on training set; applied...,7043,Prevents data leakage
8,9,T7-32,Clustering-ready file created (numeric fields ...,7043,Clustering is unsupervised and uses all records


## 9. Handover summary (T7-33)
| File | For | Content |
|---|---|---|
| `preprocessed_data/ats_clustering_ready.csv` | Dilina | 7,043 rows, 12 features (2 standardised), Churn for profiling only |
| `train_test_sets/ats_train.csv`, `ats_val.csv`, `ats_test.csv` | Shivani | 70/15/15 stratified split, 12 features + Churn, scaler fitted on train |
| `preprocessed_data/ats_encoded.csv` | Both | Encoded, unscaled version of the full dataset |
| `preprocessed_data/ats_cleaned.csv` | Team | Cleaned data with original labels |
| `train_test_sets/scaler_params.csv` | Shivani | Training mean and standard deviation for tenure and MonthlyCharges |
| `preprocessed_data/cleaning_log.csv` | Team | Every preparation step and reason |